# 24B — bounded 96×384 cache microbenchmark (CPU only)

Attach **only RSNA Knee Abnormality Detection competition data**. Set **Accelerator: None**, **Internet: Off**. Save & Run All; send `preflight_24b.json` from Output.

This notebook reads at most six report-only training studies, creates each 96×384 uint8 volume transiently, measures DICOM decode time and `np.savez_compressed` size, then discards it. It does **not** train, submit, create a full cache, or use gold labels. The representation follows the public Raptor slot/crop recipe for a capacity estimate; this is **not** a byte-identity test against its unpublished training corpus. Stop decisions require measured results and extra space for checkpoints.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import monotonic
import gc, io, json, os, shutil
import numpy as np
import pandas as pd

T0 = monotonic()
OUT = Path('/kaggle/working/preflight_24b.json')
MAX_STUDIES = 6
MAX_SECONDS = 360
IMG = 384
CROP_MM = 140.0
SLOTS = [('Sagittal', 1, 26), ('Sagittal', 0, 22), ('Coronal', 1, 18), ('Coronal', 0, 12), ('Axial', -1, 18)]
report = {'experiment': '24B_competition_only_cache_microbenchmark', 'time_utc': datetime.now(timezone.utc).isoformat(), 'limit_studies': MAX_STUDIES, 'limit_seconds': MAX_SECONDS, 'studies': [], 'warnings': [], 'cuda_used': False, 'full_cache_written': False, 'training_started': False}
root = next((p for p in (Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('/kaggle/input/rsna-knee-abnormality-detection')) if (p / 'train.csv').is_file()), None)
report['competition_root'] = str(root) if root else None
try:
    import cv2, pydicom
    from pydicom.pixel_data_handlers.util import apply_modality_lut
    dependencies_ok = True
except Exception as e:
    dependencies_ok = False
    report['warnings'].append('Dependency error: ' + type(e).__name__ + ': ' + str(e)[:200])
print('CPU-only bounded benchmark; root:', root, '| dependencies:', dependencies_ok)


In [ ]:
def header_key(path):
    try:
        d = pydicom.dcmread(str(path), stop_before_pixels=True)
        iop = getattr(d, 'ImageOrientationPatient', None)
        ipp = getattr(d, 'ImagePositionPatient', None)
        if iop is not None and ipp is not None and len(iop) == 6:
            r = np.asarray(iop[:3], dtype=float); c = np.asarray(iop[3:], dtype=float)
            pos = float(np.dot(np.asarray(ipp, dtype=float), np.cross(r, c)))
        else:
            pos = float(getattr(d, 'InstanceNumber', 0) or 0)
        spacing = getattr(d, 'PixelSpacing', None)
        spacing = float(spacing[0]) if spacing is not None else 0.5
        return pos, path, spacing
    except Exception:
        return 0.0, path, 0.5

def decode(path):
    d = pydicom.dcmread(str(path))
    a = np.asarray(apply_modality_lut(d.pixel_array, d), dtype=np.float32)
    if a.ndim != 2: raise ValueError('Expected 2-D DICOM pixel array')
    if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1': a = a.max() - a
    return a

def build_one(uid, rows):
    volume = np.zeros((96, IMG, IMG), dtype=np.uint8)
    used, offset, decoded, failed = set(), 0, 0, 0
    for plane, fluid, k in SLOTS:
        candidates = [r for r in rows if str(r.get('Anatomical_Plane', '')) == plane and str(r.get('SeriesInstanceUID', '')) not in used]
        preferred = [r for r in candidates if str(r.get('Fluid_Sensitive', '')) == str(fluid)] if fluid >= 0 else []
        chosen = preferred[0] if preferred else (candidates[0] if candidates else None)
        if chosen is None:
            offset += k; continue
        series_uid = str(chosen['SeriesInstanceUID'])
        used.add(series_uid)
        files = list((root / 'train_series' / uid / series_uid).glob('*.dcm'))
        if not files:
            offset += k; failed += k; continue
        ordered = sorted((header_key(p) for p in files), key=lambda x: x[0])
        n = len(ordered); lo = int(n * 0.02); hi = max(lo, int(n * 0.98) - 1)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else np.zeros(k, dtype=int)
        chosen_files = [ordered[min(int(i), n - 1)] for i in picks]
        arrays = []
        for _, path, spacing in chosen_files:
            try:
                arrays.append((decode(path), spacing)); decoded += 1
            except Exception:
                arrays.append((None, spacing)); failed += 1
        valid = [a for a, _ in arrays if a is not None]
        if valid:
            values = np.concatenate([a.ravel() for a in valid])
            qlo, qhi = np.percentile(values, [2, 98])
            del values
        else:
            qlo, qhi = 0.0, 1.0
        for j, (a, spacing) in enumerate(arrays):
            if a is None: continue
            a = np.clip((a - qlo) / (qhi - qlo + 1e-6), 0, 1)
            h, w = a.shape; box = min(int(round(CROP_MM / max(spacing, 1e-3))), h, w)
            y = (h - box) // 2; x = (w - box) // 2
            volume[offset + j] = (cv2.resize(a[y:y+box, x:x+box], (IMG, IMG), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
        offset += k
        del arrays, valid
    mask = np.any(volume != 0, axis=(1, 2)).astype(np.uint8)
    return volume, mask, decoded, failed


In [ ]:
if root is None:
    report['status'] = 'NEEDS_COMPETITION'
elif not dependencies_ok:
    report['status'] = 'NEEDS_DEPENDENCIES'
else:
    train = pd.read_csv(root / 'train.csv', dtype={'StudyInstanceUID': str})
    series = pd.read_csv(root / 'train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    required = {'StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive'}
    if not required.issubset(series.columns):
        report['status'] = 'MISSING_SERIES_COLUMNS'
        report['warnings'].append('Series columns: ' + str(list(series.columns)))
    else:
        labels = [c for c in train.columns if c not in ('StudyInstanceUID', 'Report')]
        weak_ids = set(train.loc[train[labels].isna().all(axis=1), 'StudyInstanceUID'].astype(str))
        group = {str(k): g.to_dict('records') for k, g in series.groupby('StudyInstanceUID', sort=False) if str(k) in weak_ids}
        ordered = sorted(group, key=lambda uid: (len(group[uid]), uid))
        ix = np.linspace(0, len(ordered) - 1, min(MAX_STUDIES, len(ordered))).round().astype(int) if ordered else []
        report['selection'] = {'eligible_weak_studies_with_series': len(ordered), 'sample_series_count': [len(group[ordered[i]]) for i in ix]}
        for i in ix:
            if monotonic() - T0 > MAX_SECONDS:
                report['warnings'].append('Time cap reached before next study')
                break
            uid = ordered[i]; t = monotonic()
            try:
                volume, mask, decoded, failed = build_one(uid, group[uid])
                b = io.BytesIO()
                np.savez_compressed(b, images=volume, mask=mask)
                row = {'study_uid': uid, 'series_count': len(group[uid]), 'valid_slices': int(mask.sum()), 'decoded_slices': decoded, 'failed_slices': failed, 'raw_bytes': int(volume.nbytes + mask.nbytes), 'compressed_npz_bytes': b.tell(), 'elapsed_seconds': round(monotonic() - t, 2)}
                del volume, mask, b
            except Exception as e:
                row = {'study_uid': uid, 'series_count': len(group[uid]), 'error': type(e).__name__ + ': ' + str(e)[:240], 'elapsed_seconds': round(monotonic() - t, 2)}
            report['studies'].append(row)
            print('study', len(report['studies']), '/', len(ix), '|', row, flush=True)
            gc.collect()
        ok = [r for r in report['studies'] if 'compressed_npz_bytes' in r and r['valid_slices'] >= 48]
        if len(ok) < len(report['studies']): report['warnings'].append('Studies with fewer than 48 valid slices or errors were excluded from capacity projection; inspect slot selection and DICOM decoding.')
        free_gib = shutil.disk_usage('/kaggle/working').free / 2**30
        report['working_free_gib'] = round(free_gib, 3)
        if ok:
            sizes = np.array([r['compressed_npz_bytes'] for r in ok], dtype=float)
            secs = np.array([r['elapsed_seconds'] for r in ok], dtype=float)
            report['projection_4349_weak_studies'] = {'median_compressed_gib': round(float(np.median(sizes) * 4349 / 2**30), 2), 'max_sample_compressed_gib': round(float(np.max(sizes) * 4349 / 2**30), 2), 'median_single_process_decode_hours': round(float(np.median(secs) * 4349 / 3600), 2), 'sample_count': len(ok), 'warning': 'Small nonrandom, series-count-spread sample; this is a rough capacity/throughput gate, NOT a full-run ETA or proof of byte-identical preprocessing.'}
        report['status'] = 'BENCHMARK_COMPLETE' if len(ok) >= 4 else 'INSUFFICIENT_SUCCESSFUL_STUDIES'
report['elapsed_seconds'] = round(monotonic() - T0, 2)
OUT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print('Wrote', OUT, '| status:', report['status'], '| elapsed:', report['elapsed_seconds'])
